# Task 4: Inspect Analytical Data Products

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-InspectDataProducts')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

ANALYTICAL_PRODUCT_LOCATION = 's3a://delta-data/analytics'
REFRESH_METADATA_PATH = (
    's3a://delta-data/metadata/analytical_product_refreshes'
)
PRODUCT_NAMES = (
    'monthly_zone_demand',
    'average_trip_distance_by_weather',
    'air_quality_vs_demand',
    'zone_weather_demand_variation',
)

## Product tables

In [6]:
for product_name in PRODUCT_NAMES:
    product_path = f'{ANALYTICAL_PRODUCT_LOCATION}/{product_name}'
    product = spark.read.format('delta').load(product_path)

    print(f'\n--- {product_name} ---')
    product.printSchema()
    product.show(10, truncate=False)


--- monthly_zone_demand ---
root
 |-- pickup_zone_name: string (nullable = true)
 |-- pickup_month: string (nullable = true)
 |-- count: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- source_path: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- refreshed_at: timestamp (nullable = true)
 |-- refresh_run_id: string (nullable = true)
 |-- schema_version: string (nullable = true)



+-----------------------+------------+-----+-------------------+------------------------------------------------+--------------------------+--------------------------+------------------------------------+--------------+
|pickup_zone_name       |pickup_month|count|product_name       |source_path                                     |created_at                |refreshed_at              |refresh_run_id                      |schema_version|
+-----------------------+------------+-----+-------------------+------------------------------------------------+--------------------------+--------------------------+------------------------------------+--------------+
|Allerton/Pelham Gardens|2024-01     |105  |monthly_zone_demand|s3a://delta-data/gold/gold_integrated_taxi_trips|2026-09-19 12:56:22.220836|2026-09-19 13:04:34.941576|c09fb9d3-3556-4a0f-ba54-a2b15c13405c|1.0           |
|Allerton/Pelham Gardens|2024-02     |94   |monthly_zone_demand|s3a://delta-data/gold/gold_integrated_taxi_trips|2026-09

## Refresh history

This table appends one row for every product refresh.

In [7]:
refresh_history = (
    spark.read.format('delta')
    .load(REFRESH_METADATA_PATH)
    .orderBy('product_name', 'refreshed_at')
)

refresh_history.show(truncate=False)

+--------------------------------+--------------------------+--------------------------+
|product_name                    |created_at                |refreshed_at              |
+--------------------------------+--------------------------+--------------------------+
|air_quality_vs_demand           |2026-09-19 12:56:38.07813 |2026-09-19 12:56:38.07813 |
|air_quality_vs_demand           |2026-09-19 12:56:38.07813 |2026-09-19 13:04:53.02837 |
|average_trip_distance_by_weather|2026-09-19 12:56:33.326626|2026-09-19 12:56:33.326626|
|average_trip_distance_by_weather|2026-09-19 12:56:33.326626|2026-09-19 13:04:48.044589|
|monthly_zone_demand             |2026-09-19 12:56:22.220836|2026-09-19 12:56:22.220836|
|monthly_zone_demand             |2026-09-19 12:56:22.220836|2026-09-19 13:04:34.941576|
|zone_weather_demand_variation   |2026-09-19 12:56:40.656657|2026-09-19 12:56:40.656657|
|zone_weather_demand_variation   |2026-09-19 12:56:40.656657|2026-09-19 13:04:56.485212|
+--------------------

In [8]:
spark.stop()